# Data Pipeline

### This notebook will clean and format the data for analysis
Februray 17th, 2025

Maxime Bouthillier

### Importing Libraries and Functions 

In [129]:
import pandas as pd
import os 
import glob 
from datetime import datetime
import warnings
from joblib import dump

# Importing Specialized Functions
import ipynb.fs.full.Pipeline_Functions as func

# Data Cleaning

### Admissions Table

In [130]:
# Setting the Directory
directory = '/Users/maxb/Library/CloudStorage/OneDrive-UniversityofWaterloo/Hospital Research/Datasets/MIMIC-III_demo'
os.chdir(directory)


# Reading the csv file
adm_df = pd.read_csv("ADMISSIONS.csv")


# Cleaning the time based features
adm_df["edregtime"] = adm_df["edregtime"].fillna("1677-09-22 00:00:00")                                             
adm_df["edouttime"] = adm_df["edouttime"].fillna("1677-09-22 00:00:00")

adm_df = func.as_datetime(adm_df, column='admittime')
adm_df = func.as_datetime(adm_df, column='dischtime')
adm_df = func.as_datetime(adm_df, column='edregtime')
adm_df = func.as_datetime(adm_df, column='edouttime')  


# Removing all observations where a patient died
adm_df = adm_df.drop(adm_df[adm_df['hospital_expire_flag'] == 1].index)


# Setting Marital Status to binary variables
adm_df['marital_status'] = adm_df['marital_status'].apply(lambda x: 1 if x == "MARRIED" else 0)

In [131]:
# Creating Readmission Feature and subsetting the dataset
adm_df = func.readmission(adm_df, 30)

# Admission Duration time feature
adm_df['admit_duration'] = adm_df['dischtime'] - adm_df['admittime']

# ED Duration time feature
adm_df['ed_duration'] = adm_df['edouttime'] - adm_df['edregtime']


# Removing unnecessary variables
col_drops = ["row_id", "language", "religion", "hospital_expire_flag", "hadm_id", 
             "has_chartevents_data", "edregtime", "edouttime", "deathtime", "diagnosis"]

for i in col_drops:
    adm_df = adm_df.drop(i, axis=1)

# Checing NaN instances
func.check_nan(adm_df)

### Patients Table

In [132]:
# Reading the csv file
pat_df = pd.read_csv("PATIENTS.csv")


# Selecting only the necessary columns
pat_df = pat_df[['subject_id', 'gender', 'dob']]


# Double checking that there are no NaN values
func.check_nan(pat_df)

### ICU LOS

In [133]:
# Reading the csv file
is_df = pd.read_csv("ICUSTAYS.csv")


# Cleaning the time based features
is_df = func.as_datetime(is_df, column='intime')
is_df = func.as_datetime(is_df, column='outtime')


# Subsetting the dataframe by only the releveat subject_ID entries:
is_df = func.subject_subset(is_df, column='intime')

In [134]:
# Keeping only the necessary columns
is_df = is_df[['subject_id', 'los']]

# Double checking that there are no NaN values
func.check_nan(is_df)

### Output Events Table

In [135]:
# Reading the csv file
oe_df = pd.read_csv("OUTPUTEVENTS.csv")


# Cleaning the time based features
oe_df = func.as_datetime(oe_df, column='charttime')


# Subsetting the dataframe by only the subject_ID entries:
oe_df = func.subject_subset(oe_df, column='charttime')

In [136]:
# Selecting only the necessary columns
oe_df = oe_df[['subject_id', # 'hadm_id', #'icustay_id', 
                    'itemid', 'value', 'valueuom', 'cgid']]

# Filling the Na Values
oe_df['value'] = oe_df['value'].fillna(0)
oe_df['valueuom'] = oe_df['valueuom'].fillna(0)

# Double checking that there are no NaN values
func.check_nan(oe_df)


### Chart Events Table

In [137]:
# Reading the csv file
ce_df = pd.read_csv("CHARTEVENTS.csv")


# Cleaning the chart time feature
ce_df = func.as_datetime(ce_df, column='charttime')


# Subsetting the dataframe by only the releveat subject_ID entries:
ce_df = func.subject_subset(ce_df, column='charttime')


# Checking for Nans
func.check_nan(ce_df)

/var/folders/l8/w77hdfv1735czt0xx90sm3_80000gn/T/ipykernel_4534/2466923477.py:2: DtypeWarning: Columns (8,10,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  ce_df = pd.read_csv("CHARTEVENTS.csv")


In [138]:
# Selecting only the necessary columns
ce_df = ce_df[['subject_id', # 'hadm_id', #'icustay_id', 
                    'itemid', 'value', 'valueuom', 'cgid']]

### Input Events CV

In [139]:
# Reading the csv file
cv_df = pd.read_csv("INPUTEVENTS_CV.csv")


# Cleaning the time based features
cv_df = func.as_datetime(cv_df, column='charttime')


# Subsetting the dataframe by only the releveat subject_ID entries:
cv_df = func.subject_subset(cv_df, column='charttime')

# Selecting only the necessary columns
cv_df = cv_df[['subject_id', 'itemid', 'amount', 'amountuom']]

/var/folders/l8/w77hdfv1735czt0xx90sm3_80000gn/T/ipykernel_4534/1016783651.py:2: DtypeWarning: Columns (17,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  cv_df = pd.read_csv("INPUTEVENTS_CV.csv")


In [140]:
# Removing any remaining rows containing NaN
cv_df = cv_df.dropna()
cols = list(cv_df.columns)

# Double checking that there are no NaN values
func.check_nan(cv_df)

### Input Events MV

In [141]:
# Reading the csv file
mv_df = pd.read_csv("INPUTEVENTS_MV.csv")


# Cleaning the time based features
mv_df = func.as_datetime(mv_df, column='starttime')


# Subsetting the dataframe by only the releveat subject_ID entries:
ie_df = func.subject_subset(mv_df, column='starttime')

In [142]:
# Selecting only the necessary columns
mv_df = mv_df[['subject_id', 'itemid', 'amount', 'amountuom']]


# Double checking that there are no NaN values
func.check_nan(mv_df)

### Lab Events

In [143]:
# Loading Dataset
labs_df = pd.read_csv("LABEVENTS.csv")


# Replacing 'abonromal' with 1, 'delta' with 2 and 'nan' with 0
labs_df['flag'] = labs_df['flag'].apply(lambda x: 1 if x == 'abnormal' else 2 if x == 'delta' else 0)


# Cleaning the time based features
labs_df = func.as_datetime(labs_df, column='charttime')


# Subsetting the dataframe by only the releveat subject_ID entries:
labs_df = func.subject_subset(labs_df, column='charttime')

In [144]:
# Dropping the unnecessary columns
col_drops = ["hadm_id", "value", "charttime", "row_id", "flag"]

for i in col_drops:
    labs_df = labs_df.drop(i, axis=1)


# Double checking that there are no NaN values
func.check_nan(labs_df)

### Prescriptions

In [145]:
# Reading the csv file
pres_df = pd.read_csv("PRESCRIPTIONS.csv")


# Cleaning the time based features
pres_df["enddate"] = pres_df["enddate"].fillna("1677-09-22 00:00:00")    
pres_df = func.as_datetime(pres_df, column='enddate')


# Subsetting the dataframe by only the releveat subject_ID entries:
pres_df = func.subject_subset(pres_df, column='enddate')


#Selecting only the relevant columns
pres_df = pres_df[['subject_id', 'drug', 'dose_val_rx','dose_unit_rx']]


# Double checking that there are no NaN values
func.check_nan(pres_df)


# Renaming
prescriptions_df = pres_df

### Procedures

In [146]:
# Reading the csv file
pro_df = pd.read_csv("PROCEDUREEVENTS_MV.csv")


# Cleaning the time based features
pro_df = func.as_datetime(pro_df, column='starttime')
pro_df = func.as_datetime(pro_df, column='endtime')


# Creating a Procedure Duration time feature
pro_df["duration"] = pro_df['endtime'] - pro_df['starttime']   


# Subsetting the dataframe by subject_IDs:
pro_df = func.subject_subset(pro_df, column='starttime')

In [147]:
# Keeping only reevant columns 
pro_df = pro_df[['subject_id','itemid','cgid','duration']]


# Removing any remaining rows containing NaN
pro_df = pro_df.dropna()
cols = list(pro_df.columns)


# Double checking that there are no NaN values
func.check_nan(pro_df)


# Renaming
procedures_df = pro_df

# Combining Tables

In [148]:
# Left join of Patients table on Admission table
master_df = pd.merge(adm_df, pat_df, how='left', on='subject_id')

# Left join of ICU LOS table on df
master_df = pd.merge(master_df, is_df, how='left', on='subject_id')

master_df = master_df.fillna(0)

In [149]:
# Combining Ouput Events and Chart Events
events_df = pd.concat([oe_df, ce_df])

# Combining Input Events CV and Input Events MV
inputs_df = pd.concat([cv_df, mv_df])

tables = [inputs_df, events_df, labs_df, prescriptions_df, procedures_df]
col_name = ['inputs', 'events', 'labs', 'prescriptions', 'procedures']

In [150]:
# Gathering subjects
subjects = list(master_df['subject_id'])

# Combining all data into one table
for i in range(len(tables)):

    new_column = []
    table = tables[i]

    for j in subjects:
        data = table.loc[table['subject_id'] ==  j]
        data = data.drop('subject_id',axis=1)
        new_column.append(data)

    master_df[col_name[i]] = new_column

dump(master_df, "Master_Dataframe.joblib")

['Master_Dataframe.joblib']

Dataframe is now cleaned and saved

In [151]:
master_df.head()

,subject_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,marital_status,ethnicity,read_flag,admit_duration,ed_duration,gender,dob,los,inputs,events,labs,prescriptions,procedures
0,42231,2102-08-29 07:15:00,2102-09-06 16:20:00,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,1,WHITE,0.0,8 days 09:05:00,0 days 00:00:00,F,2016-12-05 00:00:00,1.1358,itemid amount amountuom 10574 225...,itemid value valu...,itemid valuenum valueuom 59437 50800...,drug d...,itemid cgid duration 570 224275...
1,43881,2104-09-24 17:31:00,2104-09-30 16:17:00,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Private,1,WHITE,1.0,5 days 22:46:00,0 days 06:43:00,M,2051-03-24 00:00:00,1.9252,itemid amount amountuom 868 225154...,itemid value valueuom cgid ...,itemid valuenum valueuom 69528 51221...,dru...,itemid cgid duration 47 224275 ...
2,43879,2106-08-30 15:43:00,2106-08-31 15:15:00,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Medicare,1,BLACK/AFRICAN AMERICAN,0.0,0 days 23:32:00,0 days 00:00:00,F,2051-07-25 00:00:00,0.9775,itemid amount amountuom 10853 225...,itemid value value...,itemid valuenum valueuom 68752 50868...,drug dose_val_rx dos...,itemid cgid duration 618 224275...
3,10088,2107-01-04 11:59:00,2107-01-11 15:45:00,EMERGENCY,EMERGENCY ROOM ADMIT,SNF,Medicare,0,WHITE,1.0,7 days 03:46:00,0 days 03:51:00,M,2029-07-09 00:00:00,3.6885,itemid amount amountuom 747 30013 ...,itemid value valueuom cgid 1808 ...,itemid valuenum valueuom 16910 50861...,drug dose_va...,"Empty DataFrame Columns: [itemid, cgid, durati..."
4,10061,2107-01-16 11:33:00,2107-02-10 11:30:00,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,REHAB/DISTINCT PART HOSP,Medicare,0,WHITE,0.0,24 days 23:57:00,0 days 00:00:00,F,2031-05-19 00:00:00,24.9968,itemid amount amountuom 18 30056 ...,itemid value valueuom cgid 486 ...,itemid valuenum valueuom 11156 50862...,drug dos...,"Empty DataFrame Columns: [itemid, cgid, durati..."


In [152]:
master_df['procedures'].iloc[2]

,itemid,cgid,duration
618,224275,20889,0 days 23:19:00
619,225966,14435,0 days 00:01:00
620,225202,20889,0 days 23:12:00
621,225459,14435,0 days 00:01:00
622,225459,14576,0 days 00:01:00
623,225459,20951,0 days 00:01:00
